<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a FABRIC Facility Port

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** Demonstrates how to create a **facility port** to connect your FABRIC experiment to an **external facility** (such as Chameleon, Internet2, or a campus network). Facility ports provide dedicated Layer 2 connectivity between FABRIC nodes and external infrastructure through pre-provisioned physical links.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand what facility ports are and when to use them
2. Add a facility port to a slice using `slice.add_facility_port()`
3. Connect a facility port and a node NIC to a shared Layer 2 network
4. Manually configure IP addresses to communicate with the external facility
5. Verify connectivity to the external facility endpoint

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with Layer 2 networking on FABRIC -- see [L2 Network notebooks](../create_l2network_basic/create_l2network_basic.ipynb)
3. **Know the facility port details** for your external connection (name, site, VLAN) -- these are provisioned by FABRIC administrators

**Tip:** Facility ports must be pre-arranged with FABRIC operations. You cannot create arbitrary facility ports -- they correspond to physical connections that already exist between FABRIC sites and external facilities.

</div>

## Background: What is a Facility Port?

A **facility port** is a pre-provisioned physical network connection between a FABRIC site and an external facility. It provides dedicated Layer 2 connectivity, allowing your experiment to exchange traffic with resources outside FABRIC.

Common use cases include:
- Connecting to **Chameleon** testbed resources
- Bridging to **campus networks** or **science DMZs**
- Accessing **Internet2** or other research networks
- Integrating with **cloud providers** or **data repositories**


The facility port and your node's NIC are both connected to the same Layer 2 network, allowing direct Ethernet-level communication. A specific VLAN tag is used to isolate your traffic on the shared physical link.

## What We're Building

In this notebook we will create a FABRIC node connected to an external facility via a facility port.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Build and Submit the Slice

We create a node at the same site as the facility port and connect both to a Layer 2 network.

<div class="fab-danger">

**Important:** The node **must** be on the same FABRIC site as the facility port. Facility ports are physical connections at a specific site and cannot span sites.

</div>

In [ ]:
slice_name = "MySlice"

# --- Facility port details (pre-provisioned by FABRIC operations) ---
# These values must match an existing facility port configuration
facility_port = 'Chameleon-StarLight'
facility_port_site = 'STAR'
facility_port_vlan = '3309'

# Create a new slice
slice = fablib.new_slice(name=slice_name)

# Add a node at the SAME site as the facility port
node = slice.add_node(name=f"Node1", site='STAR')
# Add a NIC to the node -- this will connect to the L2 network
node_iface = node.add_component(model='NIC_Basic', name="nic1").get_interfaces()[0]

# Add the facility port to the slice
# The name, site, and VLAN must match the pre-provisioned facility port
facility_port = slice.add_facility_port(name=facility_port, site=facility_port_site, vlan=facility_port_vlan)
facility_port_interface = facility_port.get_interfaces()[0]

print(f"facility_port.get_site(): {facility_port.get_site()}")

# Create a Layer 2 network connecting the node NIC and facility port
net = slice.add_l2network(name=f'net_facility_port', interfaces=[])
net.add_interface(node_iface)
net.add_interface(facility_port_interface)

# Submit the slice -- blocks until provisioning is complete
slice.submit();

## Step 3: Inspect the Slice

Review the slice topology to verify that the node, network, and facility port are correctly configured.

In [ ]:
# Show slice-level information
slice.show()
# List all nodes, networks, and interfaces
slice.list_nodes()
slice.list_networks()
slice.list_interfaces()

## Step 4: Configure the Node's IP Address

Since the facility port connects to an external facility, you need to configure your node's IP address to match the external network's addressing scheme. The IP address and subnet must be coordinated with the external facility administrators.

<div class="fab-warning">

**Tip:** The IP address you use must be within the subnet agreed upon with the external facility. In this example, we use `192.168.1.0/24` and pick an IP from that range.

</div>

In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network

# Define the subnet shared with the external facility
subnet = IPv4Network("192.168.1.0/24")
# Generate a list of usable IPs (skip network and broadcast addresses)
available_ips = list(subnet)[2:]

In [ ]:
# Get the node and its interface connected to the facility port network
node1 = slice.get_node(name=f"Node1")
node1_iface = node1.get_interface(network_name=f'net_facility_port')

# Pick an IP address (index 99 = 192.168.1.101)
# Choose an IP that does not conflict with the external facility's addresses
node1_addr = available_ips.pop(99)
print(f"node1_addr: {node1_addr}")
node1_iface.ip_addr_add(addr=node1_addr, subnet=subnet)

# Verify the interface configuration
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_os_interface()}')

# Ensure the physical interface and VLAN sub-interface are up
stdout, stderr = node1.execute(f'sudo ip link set dev {node1_iface.get_physical_os_interface_name()} up')
stdout, stderr = node1.execute(f'sudo ip link set dev {node1_iface.get_os_interface()} up')

## Step 5: Test Connectivity to the External Facility

Ping the external facility's endpoint to verify the connection. The target IP (`192.168.1.10` in this example) is the address of a host on the external facility's side of the link.

In [ ]:
# Get the node and interface (useful if re-running this cell)
node1 = slice.get_node(name=f"Node1")
node1_iface = node1.get_interface(network_name=f'net_facility_port')

# Ping the external facility endpoint
stdout, stderr = node1.execute(f'ping -c 5 192.168.1.10')

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. This releases the facility port VLAN for other users.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `add_facility_port()` fails | Facility port name/site/VLAN incorrect | Verify the facility port details with FABRIC operations |
| Slice submission fails | Node not on the same site as facility port | Ensure `site` in `add_node()` matches the facility port site |
| `ping` to external host fails | IP address mismatch | Coordinate IP addressing with the external facility |
| Interface is down | Physical link not activated | Run `ip link set dev <iface> up` for both physical and VLAN interfaces |
| VLAN traffic not passing | Wrong VLAN tag | Verify the VLAN matches the pre-provisioned configuration |
| `PDP Authorization check failed` | Project permissions issue | Contact your project lead or FABRIC support |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `slice.add_facility_port(name, site, vlan)` | Add a pre-provisioned facility port | [add_facility_port](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_facility_port) |
| `facility_port.get_interfaces()` | Get the interfaces on the facility port | [get_interfaces](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.get_interfaces) |
| `facility_port.get_site()` | Get the site of the facility port | [get_site](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.get_site) |
| `slice.add_l2network(name, interfaces)` | Add a Layer 2 network | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `network.add_interface(iface)` | Attach an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `iface.get_os_interface()` | Get the OS-level interface name | [get_os_interface](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_os_interface) |
| `iface.get_physical_os_interface_name()` | Get the physical NIC interface name | [get_physical_os_interface_name](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_physical_os_interface_name) |
| `slice.list_interfaces()` | List all interfaces in the slice | [list_interfaces](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_interfaces) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **L2 Networks** | [create_l2network_basic](../create_l2network_basic/create_l2network_basic.ipynb) | Basic Layer 2 networking between FABRIC nodes |
| **FABnet IPv4 Ext** | [ipv4_ext](../create_l3network_fabnet_ipv4ext_manual/create_l3network_fabnet_ipv4ext_manual.ipynb) | Layer 3 external connectivity using public IPv4 |
| **FABnet IPv6 Ext** | [ipv6_ext](../create_l3network_fabnet_ipv6ext_manual/create_l3network_fabnet_ipv6ext_manual.ipynb) | Layer 3 external connectivity using public IPv6 |
| **Port Mirroring** | [port_mirror](../create_port_mirror/port_mirror.ipynb) | Monitor dataplane traffic with port mirroring |
| **Sub Interfaces** | [sub_interfaces](../sub_interfaces/sub_interfaces.ipynb) | Multiple virtual interfaces on a single dedicated NIC |